# Avaliação Final e Interpretabilidade

Este notebook realiza a avaliação final do modelo selecionado sobre o conjunto temporal de teste de 2024.

Após a definição do algoritmo, dos hiperparâmetros e do threshold utilizando exclusivamente os dados de 2023, os conjuntos de treino e validação são reunidos para o treinamento definitivo.

Os objetivos desta etapa são:

- treinar o modelo final utilizando todos os dados disponíveis de 2023;
- avaliar sua capacidade de generalização em 2024;
- comparar validação e teste temporal;
- analisar a importância das features;
- consolidar as limitações e conclusões do modelo.

# 1. Configuração

Nesta etapa são importadas as bibliotecas necessárias para o treinamento e avaliação final.

In [0]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
)

from pyspark.ml.classification import (
    RandomForestClassifier,
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
)

from pyspark.ml.functions import vector_to_array

# 2. Carregamento dos Dados

A base V2 e a atribuição original dos splits são carregadas novamente.

Os registros de treino e validação de 2023 serão reunidos para o treinamento final, enquanto 2024 será utilizado exclusivamente como teste temporal.

In [0]:
df_gold = spark.table("workspace.gold.ml_alfabetizacao_aluno_v2")

df_split = spark.table("workspace.gold.ml_split_modelagem")

df_modelagem = df_gold.join(
    df_split,
    on=["ano", "id_municipio"],
    how="inner",
)

In [0]:
df_desenvolvimento = df_modelagem.filter(
    F.col("split").isin(
        "treino",
        "validacao",
    )
)

df_teste = df_modelagem.filter(F.col("split") == "teste")

Validação:

In [0]:
display(
    df_modelagem.groupBy("split")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(
            F.avg(F.col("alfabetizado").cast("double")) * 100,
            2,
        ).alias("taxa_alfabetizacao"),
    )
    .orderBy("split")
)

# 3. Definição das Features

São utilizadas exatamente as mesmas features da etapa de otimização.

Nenhuma variável ou hiperparâmetro será alterado a partir dos resultados do conjunto de teste.

In [0]:
target = "alfabetizado"

features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
    "ideb_saeb_portugues_2021",
    "ideb_indicador_rendimento_2021",
]

features_numericas_imputadas = [f"{coluna}_imp" for coluna in features_numericas]

# 4. Pipeline Final

O preprocessing é reajustado utilizando todos os dados de 2023.

A Random Forest utiliza os hiperparâmetros definidos na etapa de otimização, sem qualquer ajuste baseado nos dados de 2024.

Label

In [0]:
df_desenvolvimento_ml = df_desenvolvimento.withColumn(
    "label", F.col("alfabetizado").cast("double")
)

df_teste_ml = df_teste.withColumn("label", F.col("alfabetizado").cast("double"))

Preprocessing

In [0]:
imputer = Imputer(
    inputCols=features_numericas,
    outputCols=features_numericas_imputadas,
    strategy="median",
)

rede_indexer = StringIndexer(
    inputCol="rede",
    outputCol="rede_index",
    handleInvalid="keep",
)

rede_encoder = OneHotEncoder(
    inputCols=["rede_index"],
    outputCols=["rede_ohe"],
    handleInvalid="keep",
)

assembler = VectorAssembler(
    inputCols=(features_numericas_imputadas + ["rede_ohe"]),
    outputCol="features",
)

Configuração Otimizada

In [0]:
random_forest_final = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=30,
    subsamplingRate=1.0,
    seed=42,
)

Pipeline

In [0]:
pipeline_final = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler,
        random_forest_final,
    ]
)

# 5. Treinamento do Modelo Final

O modelo definitivo é treinado utilizando todos os registros de 2023 disponíveis nos conjuntos de treino e validação.

A partir deste ponto, o conjunto temporal de 2024 é utilizado exclusivamente para avaliação de generalização.

In [0]:
modelo_final = pipeline_final.fit(df_desenvolvimento_ml)

In [0]:
pred_teste = modelo_final.transform(df_teste_ml)

## 5.1 ROC-AUC no teste temporal

In [0]:
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

auc_teste = evaluator_auc.evaluate(pred_teste)

print(f"ROC-AUC teste 2024: {auc_teste:.4f}")

# 6. Aplicação do Threshold Final

O threshold de **0,55**, definido previamente sobre a validação de 2023, é aplicado sem alterações ao conjunto de teste de 2024.

Essa etapa permite avaliar se o equilíbrio obtido na identificação dos alunos não alfabetizados se mantém fora do período utilizado no desenvolvimento.

In [0]:
pred_teste = pred_teste.withColumn("probabilidade_1", vector_to_array("probability")[1])

pred_teste_final = pred_teste.withColumn(
    "prediction",
    F.when(F.col("probabilidade_1") >= 0.55, F.lit(1.0)).otherwise(F.lit(0.0)),
)

## 6.1 Métricas Finais

In [0]:
def calcular_metricas_classificacao(df_pred):
    resumo = df_pred.agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    ).collect()[0]

    tp = resumo["tp"]
    tn = resumo["tn"]
    fp = resumo["fp"]
    fn = resumo["fn"]

    accuracy = (tp + tn) / resumo["total"]

    precision_1 = tp / (tp + fp) if (tp + fp) else 0

    recall_1 = tp / (tp + fn) if (tp + fn) else 0

    f1_1 = (
        2 * precision_1 * recall_1 / (precision_1 + recall_1)
        if (precision_1 + recall_1)
        else 0
    )

    precision_0 = tn / (tn + fn) if (tn + fn) else 0

    recall_0 = tn / (tn + fp) if (tn + fp) else 0

    f1_0 = (
        2 * precision_0 * recall_0 / (precision_0 + recall_0)
        if (precision_0 + recall_0)
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_0,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_1,
    }

In [0]:
metricas_teste = calcular_metricas_classificacao(pred_teste_final)

for metrica, valor in metricas_teste.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_teste:.4f}")

Matriz de Confusão:

In [0]:
display(
    pred_teste_final.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

# 7. Resultado no Teste Temporal de 2024

O modelo final apresentou ROC-AUC de **0,6221** no conjunto temporal de teste de 2024, ligeiramente superior ao ROC-AUC de 0,6164 observado durante a validação.

A manutenção da capacidade discriminatória em um período completamente isolado do desenvolvimento indica boa estabilidade temporal do modelo.

Com o threshold de 0,55 definido previamente na validação, foram obtidos:

- accuracy de **61,23%**;
- precision da classe `0` de **52,40%**;
- recall da classe `0` de **39,34%**;
- F1-score da classe `0` de **0,4494**;
- recall da classe `1` de **75,96%**.

Embora o recall da classe de alunos não alfabetizados tenha apresentado redução em relação à validação, o ROC-AUC não apresentou deterioração no teste temporal.

Os resultados indicam que o modelo preservou sua capacidade de discriminação em 2024, apesar das limitações já identificadas relacionadas à granularidade contextual das features.

In [0]:
resultados_generalizacao = [
    (
        "Validação 2023",
        0.6164,
        0.5966,
        0.4666,
        0.4916,
    ),
    (
        "Teste 2024",
        auc_teste,
        metricas_teste["accuracy"],
        metricas_teste["recall_0"],
        metricas_teste["f1_0"],
    ),
]

df_generalizacao = spark.createDataFrame(
    resultados_generalizacao,
    [
        "conjunto",
        "roc_auc",
        "accuracy",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_generalizacao)